# 协程

引言：协程是 Python 中更轻量的“微线程多任务方案”——线程受 GIL 限制跑不快，进程又太占资源，而协程就像线程里的“灵活小兵”，几乎不耗资源，还能在同一个线程里自己切换任务，完全听程序员的指挥，不用麻烦操作系统帮忙。虽然它运行在单线程中，却能高效应对网络请求、文件读写等 I/O 密集型场景——当一个任务等待 I/O 时，协程会立刻切换到其他任务，让线程始终保持忙碌，避免资源闲置。学会协程，能在轻量高效的前提下，进一步突破多任务处理的性能瓶颈。

# 1. 认识协程

## 1.1. 什么是协程？

协程就像“一个人同时处理多个任务”——比如你边听音乐（任务 1）、边写笔记（任务 2）、边等咖啡（任务 3）：听音乐时不用专注，写笔记累了就切换到听音乐，咖啡好了就暂停手头任务去拿，全程只有你一个人（单个线程），但多个任务并行推进。

协程的核心是“主动切换 + 状态保存”：任务执行到一定程度，主动保存当前状态（比如写笔记写到第 3 行），切换到其他任务，之后还能回到原来的状态继续执行。

## 1.2. 协程 vs 线程 vs 进程（核心区别）

| 对比维度 | 协程 | 线程 | 进程 |
|----------|------|------|------|
| 资源占用 | 极低（单个线程内切换） | 较低（共享进程资源） | 极高（独立资源空间） |
| 调度方式 | 程序员手动 / 自动切换（无系统开销） | 操作系统调度（有系统开销） | 操作系统调度（开销最高） |
| 并发能力 | 极高（支持百万级并发） | 中等（受 GIL 限制） | 中等（多核并行但数量有限） |
| 适用场景 | I/O 密集型（网络请求、文件读写） | I/O 密集型（普通并发场景） | CPU 密集型（大量计算） |
| 切换成本 | 几乎为 0（仅保存状态） | 较低（切换线程上下文） | 极高（切换进程上下文） |
| 竞争问题 | 无（单线程执行） | 有（共享资源） | 无（独立资源） |

## 1.3. 协程的核心优势

1. **轻量级**：一个线程可承载上万甚至百万个协程，资源占用远低于线程 / 进程。  

2. **高效切换**：无操作系统调度开销，切换速度比线程快 10-100 倍。  

3. **无竞争问题**：单个线程内执行，无需锁机制，避免资源竞争。  

4. **高并发**：特别适合处理大量 I/O 密集型任务（比如爬虫爬取数千个网页）。  

---


## 2. 协程基础：用生成器实现简单协程

协程的底层可以用生成器（带 `yield` 的函数）实现，通过 `next()` 或 `send()` 方法切换任务，帮你理解协程的“状态保存与切换”核心逻辑。

In [ ]:
import time
# 任务1：唱歌
def sing():
    while True:
        print("Singing")
        yield   #保存当前状态，暂停执行，切换到其他任务
        time.sleep(1)

# 任务2：跳舞
def dance():
    while True:
        print("Dancing")
        yield   #保存当前状态，暂停执行，切换到其他任务
        time.sleep(1)

if __name__ == "__main__":
    singer = sing()  # 创建唱歌任务
    dancer = dance()  # 创建跳舞任务

    while True:
        next(singer)  # 执行唱歌任务
        next(dancer)  # 执行跳舞任务        

#### 核心逻辑：`yield` 关键字负责“保存状态 + 暂停”，`next()` 负责“恢复状态 + 继续执行”，实现手动任务切换。

## 3. 协程工具 1：`greenlet`（手动切换协程）

生成器实现协程需要手动调用 `next()`，不够方便。`greenlet` 是第三方库，提供了 `switch()` 方法，能更直观地手动切换协程。

### 3.1. 安装 `greenlet`

pip install greenlet

### 3.2. greenlet 的优缺点

- **优点**：切换逻辑清晰，比生成器手动切换更简洁；  
- **缺点**：需要手动调用 `switch()` 切换，遇到 I/O 阻塞（比如 `time.sleep()`）不会自动切换，需手动处理。

## 4. 协程工具 2：`gevent`（自动切换协程，推荐）


`gevent` 是更强大的协程库，基于 `greenlet` 实现，核心优势是“遇到 I/O 操作自动切换协程”——不用手动调用 `switch()`，极大简化开发，是实际项目中最常用的协程工具。  
pip install gevent

### 4.2. 核心方法（必记）

| 方法 | 作用 |
|------|------|
| `gevent.spawn(func, *args)` | 创建协程对象并启动任务 |
| `gevent.sleep(seconds)` | 模拟 I/O 阻塞（gevent 可识别，触发自动切换） |
| 协程对象 `.join()` | 阻塞主程序，等待该协程执行完毕 |
| `gevent.joinall([协程1, 协程2])` | 等待多个协程全部执行完毕 |
| `monkey.patch_all()` | 打补丁，将 `time.sleep()` 等系统 I/O 函数替换为 `gevent` 可识别的版本 |

**关键技巧：给程序打补丁（`monkey.patch_all()`）**

实际开发中，我们常用 `time.sleep()`、`requests.get()` 等系统 I/O 函数，这些函数 `gevent` 无法直接识别，不会触发自动切换。此时需要用 `monkey.patch_all()` 打补丁，自动替换这些函数为 `gevent` 兼容版本。

## 5. 应用场景
1. **网络爬虫**：爬取大量网页，每个网页请求都是I/O阻塞(等待响应)，协程可同时发起上千个请求，大幅提高爬取效率。
2. **异步接口服务**:处理大量HTTP请求(比如API接口)，等待数据库查询、第三方服务响应时，切换到其他请求，提高并发处理能力。
3. **文件批量处理**:批量读取/写入多个文件，I/O等待时切换到其他文件操作，减少等待时间。
4. **消息队列消费**:同时消费多个消息队列的消息，I/O阻塞时切换消费其他队列，提高消费效率。

## 6. 核心总结
1. 协程本质:单个线程内的“状态保存+主动切换”，轻量级多任务方案。
2. 核心优势:资源占用极低、切换高效、无锁竞争、支持高并发(I/O密集型)。
3. 实现方式:
- 基础:生成器(`yield`)，手动切换，适合理解原理;
- 工具1:`greenlet`，手动`switch()`切换，逻辑清晰;
- 工具2:`gevent`(推荐)，自动切换I/O阻塞任务，`monkey.patch_all()`打补丁兼容系统函数。
4. 关键方法:
- `gevent.spawn()`:创建协程;
- `gevent.joinall()`:等待多个协程;
- `monkey.patch_al1()`:兼容系统I/O函数。

5. 选择技巧:
- I/O密集型、高并发场景一协程(`gevent`)
- CPU密集型场景一进程/进程池;
- 普通并发场景一线程。